# Sesión 04 — Regularización (Ridge, Lasso) y validación cruzada

**Bloque:** 2 · Regresión

## Objetivos

Al terminar esta sesión deberías ser capaz de:

- Demostrar con números por qué una sola partición train/test puede engañarte, y medir esa inestabilidad con validación cruzada.
- Elegir el número de *folds* de una validación cruzada y justificar la elección, en lugar de poner 5 por costumbre.
- Aplicar `Ridge` y `Lasso` a un problema de regresión y explicar qué le hace cada uno a los coeficientes.
- Ajustar el hiperparámetro `alpha` con `RidgeCV` y `LassoCV` en vez de a ojo, sabiendo qué se está optimizando por dentro.
- Elegir entre Ridge y Lasso a partir de lo que pide el encargo — precisión máxima o una lista corta de variables — y defender la decisión ante alguien que defienda la otra.
- Reconocer cuándo la regularización no te va a servir de nada, mirando la relación entre el número de filas y el de columnas.

## Qué necesitas de antes

- **S03 — Regresión lineal simple y múltiple**: `train_test_split`, `LinearRegression`, lectura de R², MAE y RMSE, e interpretación de coeficientes.
- La idea de que un modelo se evalúa siempre sobre datos que no ha visto.
- Del ejercicio 3 de S03 quedó una pregunta abierta: *"prueba a estandarizar las variables y compara los coeficientes"*. Hoy el escalado deja de ser opcional.

## 1. El problema: la suerte en el muestreo

En la sesión anterior partimos los datos en entrenamiento y prueba, entrenamos y medimos. Parecía riguroso. Pero esa medición depende de **una sola tirada de dados**: la de `train_test_split`. ¿Y si en el conjunto de prueba cayeron por casualidad los casos más fáciles? El error saldría artificialmente bajo. ¿Y si cayeron los más raros? Al revés.

La **validación cruzada** (*cross-validation*) sustituye esa única tirada por varias. En su versión más común, **K-Fold**:

1. Se parte el conjunto de entrenamiento en K bloques o *folds* (por ejemplo, 5).
2. Se entrena el modelo K veces. En cada una, K−1 bloques sirven para entrenar y el restante para medir.
3. Cada bloque hace de conjunto de validación exactamente una vez.
4. Se promedian las K mediciones.

Lo importante no es solo la media: es **la dispersión**. Si los K resultados se parecen entre sí, la estimación es fiable. Si bailan mucho, el modelo depende demasiado de qué filas le tocaron, y cualquier número que reportes será medio invento.

**Qué problema resuelve:** dar una estimación del error más estable que la de una sola partición, y de paso decirte cuánto te puedes fiar de ella.

**Cuándo elegirla:** casi siempre, y de forma obligada cuando el dataset es pequeño (unos pocos miles de filas o menos) o cuando vas a comparar varios modelos entre sí. Comparar dos modelos sobre una única partición es comparar dos tiradas de dados.

**Cuándo NO:** cuando entrenar es tan caro que multiplicar por K es inviable, y cuando los datos tienen orden temporal — ahí barajar las filas es hacer trampa, porque entrenarías con el futuro para predecir el pasado. Eso se ve en S22.

**Qué asume:** que las filas son independientes entre sí y que todas vienen de la misma población. Si hay grupos (varias filas del mismo cliente, del mismo hospital), un K-Fold normal reparte el mismo grupo entre entrenamiento y validación y te devuelve un resultado demasiado optimista.

> **Ojo con el vocabulario.** La validación cruzada se hace **dentro** del conjunto de entrenamiento. El conjunto de test sigue guardado y sin tocar hasta el final. Son dos cosas distintas y se mezclan constantemente.

In [ ]:
# Librerías de siempre
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Herramientas de scikit-learn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")

### El caso: precio de vivienda en Ames, Iowa

Trabajamos con el dataset **Ames Housing**: 1.460 viviendas vendidas entre 2006 y 2010, descritas con 80 columnas. Nuestro objetivo es predecir `SalePrice`.

Es un dataset deliberadamente incómodo, y por eso sirve. Tiene muchas variables, y varias miden **casi lo mismo**: `GrLivArea` (superficie habitable) frente a `1stFlrSF` + `2ndFlrSF` (superficie por planta); `GarageCars` (plazas de garaje) frente a `GarageArea` (metros de garaje); `TotRmsAbvGrd` (número de habitaciones) frente a la superficie. Un modelo lineal con variables redundantes reparte la influencia entre ellas de forma arbitraria e inestable.

Lo cargamos desde OpenML, así que la primera ejecución tarda unos segundos mientras se descarga.

In [ ]:
# Descarga desde OpenML (la primera vez tarda unos segundos; después queda cacheado)
ames = fetch_openml(data_id=42165, as_frame=True, parser="auto")
df = ames.frame

print(f"Dimensiones: {df.shape[0]} viviendas x {df.shape[1]} columnas")
print("\nAlgunas columnas representativas:")
print(df[['OverallQual', 'GrLivArea', 'YearBuilt', 'GarageCars', 'SalePrice']].head())

In [ ]:
# ¿Cuántas columnas son numéricas y cuántas son texto?
print(df.dtypes.value_counts())
print(f"\nColumnas con algún valor nulo: {(df.isnull().sum() > 0).sum()}")

### Exploración inicial (EDA)

Antes de modelar, miramos los datos. Tres preguntas:

1. ¿Cómo se distribuye `SalePrice`?
2. ¿Se nota la calidad general de la vivienda (`OverallQual`, de 1 a 10) en el precio?
3. ¿Y la superficie habitable (`GrLivArea`)?

In [ ]:
# 1. Distribución del precio de venta
plt.figure(figsize=(10, 6))
sns.histplot(df['SalePrice'], kde=True, bins=50)
plt.title('Distribución del precio de venta', fontsize=16)
plt.xlabel('Precio de venta ($)')
plt.ylabel('Frecuencia')
plt.show()

print(f"Mediana: ${df['SalePrice'].median():,.0f}   "
      f"Máximo: ${df['SalePrice'].max():,.0f}   "
      f"Asimetría: {df['SalePrice'].skew():.2f}")

In [ ]:
# 2 y 3. Dos variables candidatas frente al precio
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='OverallQual', y='SalePrice', data=df, ax=ax[0])
ax[0].set_title('Precio vs. calidad general', fontsize=14)
ax[0].set_xlabel('OverallQual (1 = muy mala, 10 = excelente)')

sns.scatterplot(x='GrLivArea', y='SalePrice', data=df, ax=ax[1], alpha=0.5)
ax[1].set_title('Precio vs. superficie habitable', fontsize=14)
ax[1].set_xlabel('GrLivArea (pies cuadrados)')

plt.tight_layout()
plt.show()

print(f"Correlación de OverallQual con el precio: {df['OverallQual'].corr(df['SalePrice']):.3f}")
print(f"Correlación de GrLivArea con el precio:   {df['GrLivArea'].corr(df['SalePrice']):.3f}")

**Qué se ve.**

La distribución del precio está **sesgada a la derecha**: muchas casas en la franja de 100.000–200.000 $ y una cola larga de casas caras. La mediana ronda los 163.000 $ y el máximo se va a 755.000 $. Es lo normal en vivienda, y tiene una consecuencia práctica: el RMSE, que penaliza los errores grandes al cuadrado, va a estar dominado por lo que hagamos con esas pocas casas caras.

`OverallQual` sube el precio de forma clara y bastante monótona, y es la variable mejor correlacionada del dataset. `GrLivArea` también sube, pero con más dispersión y con algún caso llamativo: casas enormes vendidas baratas, abajo a la derecha del gráfico.

Nada de esto nos dice todavía **qué variables sobran**, que es el problema de hoy. Para eso hacen falta las 36 numéricas a la vez, no dos.

## Preparación de los datos

Cinco pasos antes de modelar:

1. **Quedarnos con las columnas numéricas.** Las de texto (`Neighborhood`, `ExterQual`...) necesitarían codificación; las dejamos fuera hoy para no mezclar dos problemas.
2. **Rellenar los nulos** con la mediana de cada columna. `StandardScaler` no admite huecos, y la mediana aguanta bien los valores extremos.
3. **Separar `X` e `y`.**
4. **Escalar con `StandardScaler`.** Esto hoy **no es opcional**. La regularización penaliza el tamaño de los coeficientes, y el tamaño de un coeficiente depende de las unidades de su variable: un coeficiente por pie cuadrado es pequeño y uno por número de baños es grande, aunque midan efectos parecidos. Sin escalar, la penalización castigaría a unas variables mucho más que a otras solo por cómo están medidas.
5. **Inyectar ruido.** Añadimos cinco columnas de números aleatorios, sin ninguna relación con el precio. Son un control: sabemos con certeza que no sirven para nada, así que nos dicen si el modelo es capaz de darse cuenta.

In [ ]:
# 1. Solo columnas numéricas (y fuera el identificador, que no es una variable)
df_numeric = df.select_dtypes(include=np.number).copy()
df_numeric = df_numeric.drop(columns=['Id'], errors='ignore')

print(f"Nos quedamos con {df_numeric.shape[1]} columnas numéricas (SalePrice incluida)")
print("\nColumnas con nulos antes de imputar:")
print(df_numeric.isnull().sum()[lambda s: s > 0])

# 2. Nulos -> mediana de la columna
for col in df_numeric.columns:
    if df_numeric[col].isnull().any():
        df_numeric[col] = df_numeric[col].fillna(df_numeric[col].median())

print(f"\nNulos restantes: {df_numeric.isnull().sum().sum()}")

In [ ]:
# 3. Separar X e y
X = df_numeric.drop('SalePrice', axis=1)
y = df_numeric['SalePrice']

# 4. Escalado: media 0 y desviación 1 en todas las columnas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Inyección de ruido: 5 columnas aleatorias sin relación con el precio
#    La semilla fija hace que el "ruido" sea siempre el mismo y el experimento reproducible
np.random.seed(42)
ruido = np.random.rand(X_scaled.shape[0], 5)
X_scaled_ruido = np.hstack((X_scaled, ruido))

# Guardamos los nombres para poder leer los coeficientes más adelante
columnas = list(X.columns) + [f'ruido_{i + 1}' for i in range(5)]

print(f"X original:        {X_scaled.shape}")
print(f"X con ruido:       {X_scaled_ruido.shape}")
print(f"Últimas columnas:  {columnas[-7:]}")

In [ ]:
# 6. División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_ruido, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} viviendas")
print(f"Prueba:        {X_test.shape[0]} viviendas")
print(f"\nRelación filas/columnas en entrenamiento: {X_train.shape[0] / X_train.shape[1]:.0f} a 1")

### *Baseline*: regresión lineal, pero medida con validación cruzada

El modelo de la sesión anterior, sin ningún cambio. Lo único distinto es **cómo lo medimos**: en vez de una partición, cinco.

`cross_val_score` devuelve un valor por *fold*. Como scikit-learn siempre maximiza, las métricas de error se le pasan en negativo (`neg_mean_squared_error`); por eso hay que cambiarle el signo antes de la raíz.

In [ ]:
lr = LinearRegression()

# 5 folds dentro del conjunto de entrenamiento. El test no se toca.
scores_lr = cross_val_score(lr, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_lr = np.sqrt(-scores_lr)

print("----- LinearRegression -----")
print("RMSE de cada fold:")
for i, r in enumerate(rmse_lr, 1):
    print(f"  fold {i}: ${r:,.0f}")
print(f"\nRMSE promedio (CV):    ${rmse_lr.mean():,.0f}")
print(f"Desviación típica (CV): ${rmse_lr.std():,.0f}")

# Y ahora sí, el conjunto de test, una sola vez
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
rmse_lr_test = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"\nRMSE en test:          ${rmse_lr_test:,.0f}")

**Esto es lo importante de toda la primera parte de la sesión.**

Mira los cinco *folds*. No se parecen: hay uno por debajo de 27.000 $ y otro por encima de 54.000 $. La desviación típica es de unos 10.000 $ sobre una media de 37.000 — más de una cuarta parte.

Si hubiéramos hecho una sola partición y nos hubiera tocado el *fold* bueno, habríamos ido a contarle al cliente que el modelo se equivoca unos 26.000 $. Y le habríamos mentido, sin saberlo.

Ese baile tiene dos causas posibles, y conviene no confundirlas: puede que el dataset sea pequeño y heterogéneo (1.168 filas de entrenamiento, con casas muy caras repartidas de forma desigual), o puede que el modelo sea **inestable** porque tiene 41 variables, varias redundantes, y reparte su influencia de forma distinta según qué filas vea. La segunda causa sí tiene arreglo, y es lo que viene ahora.

---
### Preguntas de recap — Validación cruzada

1. ¿En qué se diferencia un *fold* de validación cruzada del conjunto de test?
2. ¿Por qué `cross_val_score` devuelve el error en negativo?
3. Si subes de `cv=5` a `cv=20`, ¿qué ganas y qué pagas?
4. Tu validación cruzada da una media excelente y una desviación típica enorme. ¿Reportarías solo la media?
5. Tienes un dataset con varias visitas del mismo paciente. ¿Qué le pasa a un K-Fold normal aquí?

---

## 2. La solución: regularización

Un modelo lineal con muchas variables tiende a **sobreajustar**: se pega tanto a los datos de entrenamiento que acaba memorizando el ruido en vez de aprender la señal. Se nota porque va muy bien en entrenamiento y peor en datos nuevos.

La **regularización** ataca esto cambiando lo que el modelo intenta minimizar. En vez de pedirle solo "equivócate lo menos posible", le pedimos dos cosas a la vez: *"equivócate poco **y además** mantén los coeficientes pequeños"*. Un coeficiente grande significa que el modelo apuesta fuerte por una variable; obligarle a apostar con moderación es lo que le impide inventarse relaciones que no existen.

Hay dos formas clásicas de medir "coeficientes pequeños", y la diferencia entre ambas no es un detalle técnico:

- **Ridge (L2)** penaliza la **suma de los cuadrados** de los coeficientes. Los encoge a todos hacia cero, pero **nunca llegan a cero exactamente**. Cuando dos variables miden casi lo mismo, Ridge reparte la influencia entre las dos en vez de escoger.
- **Lasso (L1)** penaliza la **suma de los valores absolutos**. Esta forma de penalizar sí lleva coeficientes a **cero exacto**, lo que equivale a **eliminar variables del modelo**. Cuando dos variables miden casi lo mismo, Lasso tiende a quedarse con una y tirar la otra.

El parámetro **`alpha`** controla la fuerza de la penalización. Con `alpha` = 0 vuelves a la regresión lineal de siempre. Cuanto más alto, más simple el modelo — y si te pasas, tan simple que ya no predice nada. No hay un valor bueno *a priori*: se busca, y se busca con validación cruzada. De ahí que `RidgeCV` y `LassoCV` existan.

**Qué problema resuelve:** modelos inestables por tener demasiadas variables o variables redundantes entre sí.

**Cuándo elegirla:** cuando tienes muchas columnas en relación al número de filas, cuando sospechas que hay variables que miden lo mismo, o cuando necesitas una lista corta de factores que explicar a alguien (ahí, Lasso).

**Cuándo NO:** cuando tienes muchísimas más filas que columnas y todas las variables aportan — la penalización solo te quitará precisión a cambio de nada. Lo comprobarás tú mismo en el ejercicio 3.

**Qué asume:** que las variables están **en la misma escala**. Sin escalar, la penalización castiga a cada variable según sus unidades y el resultado no significa nada. Asume además todo lo que ya asumía la regresión lineal: relación lineal y errores bien portados.

### Demostración: `LassoCV`

`LassoCV` hace dos cosas de una vez: prueba una lista de valores de `alpha` y, para cada uno, lo evalúa con validación cruzada. Se queda con el mejor y reentrena con él.

La lista de `alpha` la damos en escala logarítmica, porque lo que importa es el orden de magnitud, no la diferencia: entre 0,001 y 0,01 hay tanta distancia conceptual como entre 100 y 1.000.

In [ ]:
# Probamos 13 valores de alpha repartidos entre 0,000001 y 1.000.000
lasso_cv = LassoCV(alphas=np.logspace(-6, 6, 13), cv=5, random_state=42)
lasso_cv.fit(X_train, y_train)

print("----- LassoCV -----")
print(f"Mejor alpha encontrado: {lasso_cv.alpha_:,.6f}")

# LassoCV guarda el MSE de cada alpha en cada fold dentro de mse_path_
rmse_lasso_cv = np.sqrt(lasso_cv.mse_path_.mean(axis=1)).min()
print(f"RMSE promedio (CV):     ${rmse_lasso_cv:,.0f}")

y_pred_lasso = lasso_cv.predict(X_test)
rmse_lasso_test = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
print(f"RMSE en test:           ${rmse_lasso_test:,.0f}")

print("\n===== COMPARACIÓN =====")
print(f"LinearRegression -> RMSE test: ${rmse_lr_test:,.0f}")
print(f"LassoCV          -> RMSE test: ${rmse_lasso_test:,.0f}")

**Un resultado que no era el esperado.** Lasso **no** ha mejorado a la regresión lineal: sale unos cientos de dólares peor.

Antes de dar por fracasado el experimento, conviene mirar qué ha hecho por dentro. Ese es justo el ejercicio.

---
## Ejercicio 1 — ¿Qué ha decidido tirar Lasso?

**Contexto.** Trabajas en una consultora inmobiliaria. El equipo de tasación te pide un modelo, pero sobre todo te pide algo que puedan usar: *"dinos qué mirar cuando entramos en una casa"*. Ahora mismo tienen 41 variables sobre la mesa y no saben cuáles importan.

Acabas de entrenar un Lasso. Su promesa es que pone a cero las variables que no aportan. Vamos a comprobar si cumple, y tienes un control perfecto para hacerlo: **las cinco columnas de ruido que inyectaste tú mismo**. Sabes con total certeza que no sirven para nada. Si Lasso no las descarta, tampoco te puedes fiar del resto de su criterio.

**Tu tarea.**
1. Extrae los coeficientes del modelo (`lasso_cv.coef_`) y móntalos en un `pd.Series` usando `columnas` como índice, para poder leerlos por nombre.
2. Cuenta cuántos coeficientes son exactamente cero.
3. Comprueba **específicamente qué ha pasado con las cinco variables de ruido**.
4. Lista qué variables **reales** ha descartado. Mira si alguna te sorprende.
5. Dibuja un gráfico de barras con los 10 coeficientes positivos más grandes: los factores que, según el modelo, más suben el precio.

**Criterio de que lo has hecho bien:** las cinco de ruido tienen que estar a cero — si no, algo falla en tu código. Y el modelo debería haber descartado bastantes variables reales además del ruido, entre ellas alguna que a primera vista parecía tener sentido.

In [ ]:
# TODO: completa esta celda
# Los coeficientes salen en el mismo orden que la lista `columnas` que construiste al preparar los datos.
# Para el gráfico horizontal, ordena la serie antes de dibujarla o las barras saldrán del revés.


### Mis reflexiones

*(Escribe aquí tu respuesta antes de seguir)*

- Lasso ha descartado `GarageArea` pero ha mantenido `GarageCars`. ¿Dirías que la superficie del garaje no influye en el precio de una casa? Ten cuidado con la respuesta, y explica en qué se diferencia "no influye" de "no aporta nada nuevo".
- Las cinco variables de ruido han caído. ¿Qué habrías concluido sobre el modelo si solo hubieran caído tres? ¿Habrías podido fiarte del resto de descartes?
- El modelo predice algo peor que la regresión lineal sin regularizar, pero usa la mitad de variables. Para el encargo del equipo de tasación, ¿eso es un precio que pagarías? ¿Y si el encargo fuera fijar el precio de salida en un portal inmobiliario, sin nadie mirando?

---
### Preguntas de recap — Regularización

1. ¿Qué le hace `alpha` a los coeficientes cuando sube? ¿Y qué pasa si vale 0?
2. Ridge y Lasso encogen los coeficientes. ¿Cuál es la diferencia que de verdad importa entre los dos?
3. ¿Por qué hay que escalar las variables antes de regularizar, y qué pasaría si no lo hicieras?
4. Tienes dos variables que miden casi lo mismo. ¿Qué hace Ridge con ellas? ¿Y Lasso?
5. Un compañero dice: "Lasso puso ese coeficiente a cero, luego esa variable no afecta al precio". ¿Qué le corriges?

---

---
## Ejercicio 2 — Ridge contra Lasso, y una decisión que tomar

**Contexto.** Vuelves a la reunión con la consultora inmobiliaria, y ahora hay dos interlocutores con peticiones distintas:

- **El equipo de tasación** quiere *"los pocos factores que de verdad miramos al entrar en una casa"*, para imprimirlos en una ficha de una página.
- **El equipo de producto** quiere *"el modelo más preciso posible"* para calcular un precio de salida automático en la web. Le da igual cuántas variables use; nadie va a leerlo.

Has probado Lasso. Falta Ridge. Al terminar tienes que entregar **una recomendación para cada equipo**, y sostenerla.

**Tu tarea.**
1. Entrena un `RidgeCV` con la misma rejilla de `alphas` y los mismos datos de entrenamiento. Anota el `alpha` que elige.
2. Evalúa su RMSE en test y ponlo junto a los otros dos modelos en una única comparación.
3. Extrae sus coeficientes y cuenta cuántos son exactamente cero.
4. Mira específicamente qué ha hecho Ridge con las cinco variables de ruido, y compáralo con lo que hizo Lasso.
5. Construye un gráfico de barras que enfrente los coeficientes de Lasso y Ridge para las 15 variables más importantes en valor absoluto. Busca los casos donde uno dice cero y el otro no.

**Criterio de que lo has hecho bien:** los tres modelos deberían quedar muy cerca en RMSE — la decisión no se va a poder tomar mirando solo esa columna, y eso es parte del ejercicio. En el gráfico tienen que verse variables donde Lasso marca cero y Ridge mantiene un coeficiente apreciable: ahí es donde se ve la diferencia entre las dos penalizaciones.

In [ ]:
# TODO: completa esta celda
# `RidgeCV` se usa igual que `LassoCV` pero no necesita `cv` ni `random_state`: trae su propia validación interna.
# Para el recuento de ceros no busques valores pequeños, busca ceros exactos: son cosas distintas y ahí está el quid.


In [ ]:
# TODO: completa esta celda
# Para ordenar por importancia sin que el signo estorbe, usa el valor absoluto solo para elegir el orden, no para dibujar.
# La comparación interesante no es cuál tiene el coeficiente más grande, sino en qué filas uno vale exactamente 0 y el otro no.


### Mis reflexiones

*(Escribe aquí tu respuesta)*

- Has recomendado un modelo a cada equipo. Escribe, en dos frases y sin tecnicismos, lo que le dirías a cada uno. Ahora imagina que se sientan juntos y comparan lo que les has dicho: ¿te sostiene el argumento?
- El coeficiente de `GrLivArea` es aproximadamente el doble en Lasso que en Ridge. ¿Significa eso que un modelo cree que la superficie importa el doble que el otro? Si no, ¿qué significa?
- Los tres modelos se diferencian en menos del 1 % de RMSE. ¿Qué otra información pedirías antes de decidir, que no esté en este notebook?
- ¿Qué tendría que pasar para que cambiaras de recomendación? Da una condición concreta y comprobable, no "si los datos fueran distintos".

---
## Ejercicio 3 — Cuando la regularización no sirve de nada

**Contexto.** Te cambian de proyecto. Ahora es una inmobiliaria que opera en California y el dataset es **California Housing**: 20.640 distritos, pero solo **8 variables** numéricas (longitud, latitud, antigüedad de la vivienda, habitaciones, dormitorios, población, hogares e ingreso medio). El objetivo es `median_house_value`.

Tu jefe de proyecto, que estuvo en la reunión anterior, te dice: *"aplica lo mismo que en Ames, que funcionó muy bien"*.

**Tu tarea.**
1. Carga California Housing desde la URL que tienes abajo, quédate con las columnas numéricas e imputa los nulos con la mediana.
2. Inyecta las mismas **5 columnas de ruido** que usaste en clase.
3. **Escala correctamente**: parte primero en train y test, y ajusta el `StandardScaler` **solo con los datos de entrenamiento**. Fíjate en que esto **no** es lo que hicimos en clase, y guárdate la observación para la última reflexión.
4. Entrena `LinearRegression`, `LassoCV` y `RidgeCV`, y compara sus RMSE en test.
5. Cuenta cuántas variables descarta Lasso aquí, incluidas las de ruido.
6. Calcula la relación filas/columnas en los dos datasets, Ames y California, y ponlas una al lado de la otra.

**Criterio de que lo has hecho bien:** los tres modelos van a quedar prácticamente empatados, mucho más que en Ames, y Lasso apenas va a descartar nada. Si te sale así, no te has equivocado: **el resultado del ejercicio es que la técnica no aporta**. Lo que se evalúa es que sepas explicar por qué, y que el número del punto 6 aparezca en tu explicación.

**Entrega:** el notebook con las celdas completadas y las reflexiones escritas.

In [ ]:
# TODO: completa esta celda
# El orden de los pasos importa: `train_test_split` primero y `StandardScaler().fit()` solo sobre el train, después.
# Cuando compares los dos datasets, no mires el número de filas ni el de columnas por separado: mira su cociente.


### Mis reflexiones finales

*(Escribe aquí tu respuesta)*

- Tu jefe te dijo "aplica lo mismo que en Ames, que funcionó muy bien". Escríbele la respuesta: tres o cuatro frases, sin jerga, explicando qué has hecho y por qué aquí no aporta. No vale decir solo "el dataset es distinto".
- Da una regla práctica que puedas aplicar **antes** de entrenar nada para sospechar si la regularización te va a servir en un problema nuevo. Tiene que ser algo que puedas calcular mirando el dataset, y tiene que poder fallar: si tu regla acierta siempre, es que no dice nada.
- Compara el orden de los pasos de preparación en este ejercicio y en los dos anteriores. Hay una diferencia deliberada. Encuéntrala, explica por qué una de las dos formas es incorrecta y razona si en este caso concreto habría cambiado mucho el resultado.
- Has visto tres modelos empatados en Ames y tres modelos empatados en California, por motivos distintos en cada caso. ¿En qué se diferencian los dos empates?
- Si tuvieras que llevar a producción el modelo de Ames, ¿qué añadirías a este notebook antes de que alguien lo use sin ti delante?